<a href="https://colab.research.google.com/github/raw-fun/Colab-Script/blob/main/V4_Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title 🚀 Step 1: Initialize System & Fonts
import subprocess, sys, os, io, textwrap, time, json as _json, copy, math
from collections import deque
from IPython.display import display, clear_output, HTML

# Install missing packages
def install_and_import():
    packages = {"ipywidgets": "ipywidgets", "pillow": "PIL", "requests": "requests", "qrcode": "qrcode"}
    for pkg, imp in packages.items():
        try:
            __import__(imp)
        except ImportError:
            print(f"⏳ Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_and_import()

import ipywidgets as widgets
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import requests
import qrcode

# Font Management
class FontManager:
    _fonts = {}
    _loaded = False

    @classmethod
    def load_fonts(cls):
        if cls._loaded: return cls._fonts
        urls = {
            "Bold":    "https://github.com/google/fonts/raw/main/ofl/hindsiliguri/HindSiliguri-Bold.ttf",
            "Medium":  "https://github.com/google/fonts/raw/main/ofl/hindsiliguri/HindSiliguri-Medium.ttf",
            "Regular": "https://github.com/google/fonts/raw/main/ofl/hindsiliguri/HindSiliguri-Regular.ttf",
        }
        os.makedirs("fonts", exist_ok=True)
        print("🔤 Loading fonts...")
        for name, url in urls.items():
            path = f"fonts/Font_{name}.ttf"
            if not os.path.exists(path):
                print(f"  ⏳ Downloading {name}...")
                r = requests.get(url, timeout=15)
                with open(path, "wb") as f: f.write(r.content)
            cls._fonts[name] = path
        cls._loaded = True
        return cls._fonts

FONTS = FontManager.load_fonts()
PRIMARY_FONT   = FONTS["Bold"]
SECONDARY_FONT = FONTS["Medium"]
REGULAR_FONT   = FONTS["Regular"]
FONT_MAP = {"Regular": REGULAR_FONT, "Medium": SECONDARY_FONT, "Bold": PRIMARY_FONT}

print("✅ Step 1: System and Fonts Ready!")

⏳ Installing qrcode...
🔤 Loading fonts...
  ⏳ Downloading Bold...
  ⏳ Downloading Medium...
  ⏳ Downloading Regular...
✅ Step 1: System and Fonts Ready!


In [2]:
# @title 🛠️ Step 2: Advanced Graphic Utilities
def hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def create_gradient(width, height, start_hex, end_hex, direction="vertical"):
    img  = Image.new("RGB", (width, height))
    draw = ImageDraw.Draw(img)
    s, e = hex_to_rgb(start_hex), hex_to_rgb(end_hex)
    if direction == "vertical":
        for i in range(height):
            t = i / max(height-1, 1)
            c = tuple(int(s[j]+(e[j]-s[j])*t) for j in range(3))
            draw.line([(0,i),(width,i)], fill=c)
    elif direction == "horizontal":
        for i in range(width):
            t = i / max(width-1, 1)
            c = tuple(int(s[j]+(e[j]-s[j])*t) for j in range(3))
            draw.line([(i,0),(i,height)], fill=c)
    elif direction == "diagonal":
        steps = width + height
        for i in range(steps):
            t = i / max(steps-1, 1)
            c = tuple(int(s[j]+(e[j]-s[j])*t) for j in range(3))
            draw.line([(max(0,i-height), min(i,height)),
                       (min(i,width),   max(0,i-width))], fill=c)
    return img

def draw_text_with_shadow(draw, text, font, fill, x, y, shadow_color="#000000", offset=(3,3)):
    draw.text((x+offset[0], y+offset[1]), text, font=font, fill=shadow_color)
    draw.text((x, y), text, font=font, fill=fill)

def get_text_x(text, font, align="left", margin_x=80, canvas_w=1080):
    try:    tw = font.getlength(text)
    except: tw = len(text) * max(font.size // 2, 1)
    if align == "center": return int((canvas_w - tw) // 2)
    if align == "right":  return int(canvas_w - margin_x - tw)
    return margin_x

def draw_text_wrapped(draw, text, font, color, x, y, max_width, line_spacing=10):
    try:    cw = max(font.getlength("A"), 1)
    except: cw = max(font.size // 2, 1)
    chars = max(1, int(max_width / cw))
    lines = textwrap.wrap(text, width=chars) or [text]
    cur_y = y
    for line in lines:
        draw.text((x, cur_y), line, font=font, fill=color)
        bbox  = draw.textbbox((0,0), line, font=font)
        cur_y += (bbox[3]-bbox[1]) + line_spacing
    return cur_y

def draw_accent_box(draw, x1, y1, x2, y2, radius, bg_color, border_color=None, border_width=3):
    draw.rounded_rectangle([(x1,y1),(x2,y2)], radius=radius, fill=bg_color)
    if border_color:
        draw.rounded_rectangle([(x1,y1),(x2,y2)], radius=radius, outline=border_color, width=border_width)

def apply_pattern_overlay(base_img, pattern_type, hex_color, opacity, spacing=40):
    w, h   = base_img.size
    overlay = Image.new("RGBA", (w, h), (0,0,0,0))
    drw     = ImageDraw.Draw(overlay)
    r,g,b   = hex_to_rgb(hex_color)
    alpha   = int(255 * min(max(opacity, 0), 1))
    c       = (r, g, b, alpha)
    sp      = max(spacing, 5)

    if pattern_type == "dots":
        rad = max(2, sp // 10)
        for y in range(0, h+sp, sp):
            for x in range(0, w+sp, sp):
                drw.ellipse([(x-rad,y-rad),(x+rad,y+rad)], fill=c)
    elif pattern_type == "grid":
        for y in range(0, h, sp):
            drw.line([(0,y),(w,y)], fill=c, width=1)
        for x in range(0, w, sp):
            drw.line([(x,0),(x,h)], fill=c, width=1)
    elif pattern_type == "diagonal":
        for i in range(-(h+sp), w+h+sp, sp):
            drw.line([(i,0),(i+h+sp,h+sp)], fill=c, width=1)
    elif pattern_type == "hexagon":
        hw = sp
        hh = max(int(sp * 0.866), 1)
        for row in range(-1, h//hh+2):
            for col in range(-1, w//hw+2):
                ox  = (hw//2) if row%2 else 0
                cx2 = col*hw + ox
                cy2 = row*hh
                pts = [(cx2+int((hw//2)*math.cos(math.radians(a))),
                        cy2+int((hw//2)*math.sin(math.radians(a))))
                       for a in range(0,360,60)]
                drw.polygon(pts, outline=c)

    return Image.alpha_composite(base_img.convert("RGBA"), overlay).convert("RGB")

def apply_bg_image(base_img, pil_img, opacity):
    if pil_img is None: return base_img
    w, h = base_img.size
    bg   = pil_img.resize((w,h), Image.LANCZOS).convert("RGBA")
    r,g,b,a = bg.split()
    a = a.point(lambda p: int(p * min(max(opacity,0),1)))
    bg.putalpha(a)
    return Image.alpha_composite(base_img.convert("RGBA"), bg).convert("RGB")

def apply_logo(base_img, logo_pil, size, position, opacity):
    if logo_pil is None: return base_img
    w, h   = base_img.size
    lw, lh = logo_pil.size
    ratio  = size / max(lw, lh, 1)
    ns     = (max(1,int(lw*ratio)), max(1,int(lh*ratio)))
    logo   = logo_pil.resize(ns, Image.LANCZOS).convert("RGBA")
    r,g,b,a = logo.split()
    a = a.point(lambda p: int(p * min(max(opacity,0),1)))
    logo.putalpha(a)
    lw2, lh2 = logo.size
    pad = 30
    pos_map = {
        "top-left":     (pad, pad),
        "top-right":    (w-lw2-pad, pad),
        "bottom-left":  (pad, h-lh2-pad),
        "bottom-right": (w-lw2-pad, h-lh2-pad),
    }
    pos = pos_map.get(position, (pad, pad))
    base_rgba = base_img.convert("RGBA")
    base_rgba.paste(logo, pos, logo)
    return base_rgba.convert("RGB")

def apply_box_shadow(img, x1, y1, x2, y2, radius, offset=12, blur=10):
    w, h = img.size
    shadow = Image.new("RGBA", (w,h), (0,0,0,0))
    sd     = ImageDraw.Draw(shadow)
    sd.rounded_rectangle([(x1+offset//2, y1+offset), (x2+offset//2, y2+offset)], radius=radius, fill=(0,0,0,110))
    shadow = shadow.filter(ImageFilter.GaussianBlur(blur))
    return Image.alpha_composite(img.convert("RGBA"), shadow).convert("RGB")

def draw_badge_shape(draw, cx, cy, radius, shape, fill_color, border_color=None, border_width=2):
    if shape == "circle":
        draw.ellipse([(cx-radius,cy-radius),(cx+radius,cy+radius)], fill=fill_color, outline=border_color, width=border_width if border_color else 0)
    elif shape == "square":
        r = int(radius*0.85)
        draw.rounded_rectangle([(cx-r,cy-r),(cx+r,cy+r)], radius=10, fill=fill_color, outline=border_color, width=border_width if border_color else 0)
    elif shape == "diamond":
        pts = [(cx, cy-radius),(cx+radius, cy),(cx, cy+radius),(cx-radius, cy)]
        draw.polygon(pts, fill=fill_color, outline=border_color)

def draw_divider_adv(img, x1, x2, y, color, style="solid", thickness=2, color2=None):
    draw = ImageDraw.Draw(img)
    if style == "solid":
        draw.line([(x1,y),(x2,y)], fill=color, width=thickness)
    elif style == "dotted":
        dash, x = 15, x1
        while x < x2:
            draw.line([(x,y),(min(x+dash,x2),y)], fill=color, width=thickness)
            x += dash*2
    elif style == "double":
        draw.line([(x1,y),(x2,y)], fill=color, width=1)
        draw.line([(x1,y+4),(x2,y+4)], fill=color, width=1)
    elif style == "gradient":
        s, e = hex_to_rgb(color), hex_to_rgb(color2 or "#000000")
        st = x2 - x1
        for i in range(st):
            t = i / max(st-1, 1)
            c = tuple(int(s[j]+(e[j]-s[j])*t) for j in range(3))
            draw.line([(x1+i,y),(x1+i,y+thickness-1)], fill=c)
    return img

def get_uploaded_image(upload_widget):
    if not upload_widget.value: return None
    try:
        val = upload_widget.value
        content = list(val.values())[0]["content"] if isinstance(val, dict) else val[0]["content"]
        return Image.open(io.BytesIO(content)).convert("RGBA")
    except Exception: return None

# --- NEW V4 UTILITIES ---
def apply_glassmorphism(base_img, x1, y1, x2, y2, radius, blur_radius=15, fill_color="#FFFFFF", opacity=40):
    mask = Image.new("L", base_img.size, 0)
    draw_mask = ImageDraw.Draw(mask)
    draw_mask.rounded_rectangle([(x1, y1), (x2, y2)], radius=radius, fill=255)

    region = base_img.crop((x1, y1, x2, y2))
    region = region.filter(ImageFilter.GaussianBlur(blur_radius))

    r,g,b = hex_to_rgb(fill_color)
    overlay = Image.new("RGBA", region.size, (r, g, b, int(2.55 * opacity)))
    region = Image.alpha_composite(region.convert("RGBA"), overlay)

    base_rgba = base_img.convert("RGBA")
    temp_img = Image.new("RGBA", base_img.size)
    temp_img.paste(region, (x1, y1))

    return Image.composite(temp_img, base_rgba, mask).convert("RGB")

def add_noise(img, opacity=0.05):
    import random
    w, h = img.size
    noise = Image.new("RGBA", (w, h), (0,0,0,0))
    pixels = noise.load()
    alpha = int(255 * opacity)
    for y in range(h):
        for x in range(w):
            val = random.randint(0, 255)
            pixels[x, y] = (val, val, val, alpha)
    return Image.alpha_composite(img.convert("RGBA"), noise).convert("RGB")

def generate_qr(url, size=150, color="#FFFFFF", bg_color=None):
    qr = qrcode.QRCode(box_size=10, border=2)
    qr.add_data(url)
    qr.make(fit=True)
    img_qr = qr.make_image(fill_color=color, back_color=bg_color or (0,0,0,0)).convert("RGBA")
    return img_qr.resize((size, size), Image.LANCZOS)

print("✅ Step 2: Advanced Graphic Utilities Ready!")

✅ Step 2: Advanced Graphic Utilities Ready!


In [3]:
# @title 🎨 Step 3: Advanced Rendering Engine (v4)
def render_template(vals, bg_img, logo_img):
    try:
        t0 = time.time()

        # Fonts mapping
        f_head    = ImageFont.truetype(FONT_MAP.get(vals.get("head_font", "Regular"), REGULAR_FONT), vals.get("head_sz", 20))
        f_h1      = ImageFont.truetype(FONT_MAP.get(vals.get("hero_font", "Bold"), PRIMARY_FONT), vals.get("hero1_sz", 90))
        f_h2      = ImageFont.truetype(FONT_MAP.get(vals.get("hero_font", "Bold"), PRIMARY_FONT), vals.get("hero2_sz", 90))
        f_h3      = ImageFont.truetype(FONT_MAP.get(vals.get("hero_font", "Bold"), PRIMARY_FONT), vals.get("hero3_sz", 70))
        f_bdgn    = ImageFont.truetype(PRIMARY_FONT, vals.get("bdg_sz", 24))
        f_bdgt    = ImageFont.truetype(PRIMARY_FONT, 30)
        f_bt      = ImageFont.truetype(FONT_MAP.get(vals.get("body_font", "Medium"), PRIMARY_FONT), 24)
        f_btxt    = ImageFont.truetype(FONT_MAP.get(vals.get("body_font", "Medium"), SECONDARY_FONT), 28)
        f_foot    = ImageFont.truetype(REGULAR_FONT, vals.get("foot_sz", 22))
        f_stn     = ImageFont.truetype(PRIMARY_FONT, 64)
        f_stl     = ImageFont.truetype(SECONDARY_FONT, 22)

        cv_w, cv_h = vals["cv_w"], vals["cv_h"]
        margin_x = vals["margin_x"]

        # 1. Canvas & Gradient
        img = create_gradient(cv_w, cv_h, vals["bg_start"], vals["bg_end"], direction=vals["grad_dir"])

        # 2. BG Image
        if bg_img is not None:
            img = apply_bg_image(img, bg_img, vals["bg_img_opacity"])

        # 3. Pattern Overlay
        if vals.get("show_pattern"):
            img = apply_pattern_overlay(img, vals["pattern_type"], vals["pattern_col"], vals["pattern_opacity"], vals["pattern_spacing"])

        # 4. Noise/Grain (V4 Feature)
        if vals.get("use_noise"):
            img = add_noise(img, opacity=vals.get("noise_opacity", 0.04))

        draw = ImageDraw.Draw(img)

        # 5. Top Accent Bar
        if vals.get("show_accent_bar") and vals.get("accent_bar_h", 0) > 0:
            draw.rectangle([(0,0),(cv_w, vals["accent_bar_h"])], fill=vals["accent_bar_col"])

        # 6. Header
        hx = get_text_x(vals["head_txt"], f_head, vals["head_align"], margin_x, cv_w)
        draw.text((hx, vals["head_y"]), vals["head_txt"], font=f_head, fill=vals["head_col"])

        # 7. Hero Texts
        hero_align = vals.get("hero_align", "left")
        hero_shadow = vals.get("hero_shadow", True)

        for txt, fnt, col, y_pos in [
            (vals["hero1_txt"], f_h1, vals["hero1_col"], vals["hero1_y"]),
            (vals["hero2_txt"], f_h2, vals["hero2_col"], vals["hero2_y"]),
        ]:
            if txt.strip():
                tx = get_text_x(txt, fnt, hero_align, margin_x, cv_w)
                if hero_shadow:
                    draw_text_with_shadow(draw, txt, fnt, col, tx, y_pos, shadow_color="#000033", offset=(4,4))
                else:
                    draw.text((tx, y_pos), txt, font=fnt, fill=col)

        if vals.get("show_hero3") and vals["hero3_txt"].strip():
            h3x = get_text_x(vals["hero3_txt"], f_h3, hero_align, margin_x, cv_w)
            if hero_shadow:
                draw_text_with_shadow(draw, vals["hero3_txt"], f_h3, vals["hero3_col"], h3x, vals["hero3_y"], shadow_color="#000033", offset=(4,4))
            else:
                draw.text((h3x, vals["hero3_y"]), vals["hero3_txt"], font=f_h3, fill=vals["hero3_col"])

        # 8. Divider
        if vals.get("show_divider"):
            last_y = (vals["hero3_y"] + vals["hero3_sz"] if (vals.get("show_hero3") and vals["hero3_txt"].strip()) else vals["hero2_y"] + vals["hero2_sz"])
            div_y = last_y + 20
            img = draw_divider_adv(img, margin_x, cv_w-margin_x, div_y, vals["divider_col"], style=vals.get("divider_style", "solid"), thickness=2, color2=vals.get("bg_end"))
            draw = ImageDraw.Draw(img)

        # 9. Multi-Module Box (V4 Feature: Glassmorphism & Multiple Modules)
        if vals.get("show_mod"):
            mod_count = vals.get("module_count", 1)
            base_y = vals["mod_y"]
            spacing_y = 30
            mod_h = vals["mod_h"]
            x1, x2 = margin_x, cv_w - margin_x

            for i in range(mod_count):
                y1 = base_y + i * (mod_h + spacing_y)
                y2 = y1 + mod_h

                if vals.get("mod_shadow"):
                    img = apply_box_shadow(img, x1, y1, x2, y2, vals["mod_rad"])
                    draw = ImageDraw.Draw(img)

                if vals.get("use_glass"):
                    img = apply_glassmorphism(img, x1, y1, x2, y2, radius=vals["mod_rad"], blur_radius=20, fill_color=vals["mod_bg"], opacity=vals.get("glass_opacity", 0.3))
                    draw = ImageDraw.Draw(img)
                    if vals.get("show_mod_border"):
                        draw.rounded_rectangle([(x1,y1),(x2,y2)], radius=vals["mod_rad"], outline=vals["mod_border_col"], width=3)
                else:
                    border_c = vals["mod_border_col"] if vals.get("show_mod_border") else None
                    draw_accent_box(draw, x1, y1, x2, y2, vals["mod_rad"], vals["mod_bg"], border_color=border_c, border_width=3)

                # Badge
                bx, by   = x1+40, y1+40
                bdg_num = f"0{i+1}" if mod_count > 1 else vals.get("bdg_num", "01")

                draw_badge_shape(draw, bx+15, by+15, 30, vals.get("bdg_shape", "circle"), "#1E293B", border_color=vals["bdg_col"], border_width=2)
                draw.text((bx-8, by-4), bdg_num, font=f_bdgn, fill=vals["bdg_col"])
                draw.text((bx+60, by-5), vals["bdg_title"], font=f_bdgt, fill="white")

                # Inner Content
                cur_y = by + 70
                draw.text((bx, cur_y), vals["err_title"], font=f_bt, fill=vals["err_col"])
                cur_y += 35
                cur_y = draw_text_wrapped(draw, vals["err_txt"], f_btxt, "#B0BEC5", bx+20, cur_y, (x2-x1)-60)

                draw.line([(bx, cur_y+vals.get("spacing", 45)//2),(x2-40, cur_y+vals.get("spacing", 45)//2)], fill="#2D3748", width=1)

                cur_y += vals.get("spacing", 45)+10
                draw.text((bx, cur_y), vals["opt_title"], font=f_bt, fill=vals["opt_col"])
                cur_y += 35
                draw_text_wrapped(draw, vals["opt_txt"], f_btxt, "white", bx+20, cur_y, (x2-x1)-60)

        # 10. Stats Block
        if vals.get("show_stats"):
            items = [(vals["stats_num1"],vals["stats_lbl1"]), (vals["stats_num2"],vals["stats_lbl2"]), (vals["stats_num3"],vals["stats_lbl3"])]
            valid = [(n,l) for n,l in items if n.strip()]
            if valid:
                uw = cv_w - 2*margin_x
                col_w = uw // len(valid)
                for i,(num,lbl) in enumerate(valid):
                    cx = margin_x + i*col_w + col_w//2
                    try:    nw = int(f_stn.getlength(num))
                    except: nw = len(num)*35
                    try:    lw2 = int(f_stl.getlength(lbl))
                    except: lw2 = len(lbl)*12
                    draw_text_with_shadow(draw, num, f_stn, vals["stats_col"], cx-nw//2, vals["stats_y"], shadow_color="#000000", offset=(3,3))
                    draw.text((cx-lw2//2, vals["stats_y"]+78), lbl, font=f_stl, fill="#B0BEC5")

        # 11. Footer
        if vals.get("show_footer") and vals["foot_txt"].strip():
            fy = cv_h - vals["foot_sz"] - 40
            draw.line([(margin_x, fy-20),(cv_w-margin_x, fy-20)], fill="#2D3748", width=1)
            fx = get_text_x(vals["foot_txt"], f_foot, "center", margin_x, cv_w)
            draw.text((fx, fy), vals["foot_txt"], font=f_foot, fill=vals["foot_col"])

        # 12. Logo / Watermark
        if vals.get("show_logo") and logo_img is not None:
            img = apply_logo(img, logo_img, vals["logo_size"], vals["logo_pos"], vals["logo_opacity"])

        # 13. QR Code (V4 Feature)
        if vals.get("show_qr") and vals.get("qr_url"):
            qr_img = generate_qr(vals["qr_url"], size=vals["qr_size"], color=vals["qr_col"])
            qx, qy = cv_w - vals["qr_size"] - margin_x, cv_h - vals["qr_size"] - 100
            if vals.get("show_footer"): qy -= 60
            img.convert("RGBA").paste(qr_img, (qx, qy), qr_img)

        ms = (time.time()-t0)*1000
        return img, ms

    except Exception as e:
        import traceback; traceback.print_exc()
        err_img = Image.new("RGB",(600,200),"#3B0000")
        ImageDraw.Draw(err_img).text((10,80), f"Render Error: {e}", fill="white")
        return err_img, 0

print("✅ Step 3: Advanced Rendering Engine (v4) Ready!")

✅ Step 3: Advanced Rendering Engine (v4) Ready!


In [4]:
# @title 🎛️ Step 4: UI Controls – Canvas, Effects & Header
style       = {"description_width": "160px"}
layout_full = widgets.Layout(width="98%")
layout_half = widgets.Layout(width="49%")
layout_btn  = widgets.Layout(width="140px", height="35px")

# ══ 1. CANVAS ════════════════════════════════════════════════════
w_cv_w     = widgets.IntSlider(value=1080, min=800, max=2000, step=10, description="প্রস্থ:", style=style, layout=layout_full)
w_cv_h     = widgets.IntSlider(value=1350, min=800, max=2000, step=10, description="উচ্চতা:", style=style, layout=layout_full)
w_bg_start = widgets.ColorPicker(value="#050814", description="BG শুরু:", style=style)
w_bg_end   = widgets.ColorPicker(value="#1A1030", description="BG শেষ:", style=style)
w_grad_dir = widgets.ToggleButtons(
    options=["vertical","horizontal","diagonal"], value="vertical",
    description="Gradient:", style={"description_width":"160px","button_width":"100px"})

# Aspect Ratio Presets
ASPECT_PRESETS = {
    "📱 Story 9:16": (1080,1920), "⬛ Post 1:1": (1080,1080),
    "💼 LinkedIn":   (1200, 628), "🎬 YT Thumb": (1280, 720),
    "📌 Pinterest":  (1000,1500), "🐦 Twitter":  (1600, 900),
}
def _mk_aspect(w,h):
    def _h(b): w_cv_w.value=w; w_cv_h.value=h
    return _h
_aspect_btns = []
for name,(w,h) in ASPECT_PRESETS.items():
    b = widgets.Button(description=name, layout=layout_btn)
    b.on_click(_mk_aspect(w,h))
    _aspect_btns.append(b)

# ══ 2. V4 EFFECTS & BG IMAGE ═════════════════════════════════════
w_use_noise     = widgets.Checkbox(value=False, description="Noise/Grain Effect", style=style)
w_noise_opacity = widgets.FloatSlider(value=0.04, min=0.01, max=0.15, step=0.01, description="Noise Opacity:", style=style)

w_bg_img_upload  = widgets.FileUpload(accept="image/*", multiple=False, description="BG Image Upload:", layout=widgets.Layout(width="300px"))
w_bg_img_opacity = widgets.FloatSlider(value=0.40, min=0.0, max=1.0, step=0.05, description="BG Opacity:", style=style)

# ══ 3. BACKGROUND PATTERN ════════════════════════════════════════
w_show_pattern    = widgets.Checkbox(value=False, description="Pattern দেখান", style=style)
w_pattern_type    = widgets.ToggleButtons(
    options=["dots","grid","diagonal","hexagon"], value="dots",
    description="Pattern:", style={"description_width":"80px","button_width":"90px"})
w_pattern_col     = widgets.ColorPicker(value="#FFFFFF", description="Pattern রঙ:", style=style)
w_pattern_opacity = widgets.FloatSlider(value=0.08, min=0.02, max=0.40, step=0.01, description="Opacity:", style=style)
w_pattern_spacing = widgets.IntSlider(value=40, min=15, max=120, description="Spacing:", style=style)

# ══ 4. TOP ACCENT BAR ════════════════════════════════════════════
w_show_accent_bar = widgets.Checkbox(value=True, description="Top Accent Bar", style=style)
w_accent_bar_h    = widgets.IntSlider(value=8, min=2, max=60, description="Bar উচ্চতা:", style=style)
w_accent_bar_col  = widgets.ColorPicker(value="#00E5FF", description="Bar রঙ:", style=style)

# ══ 5. HEADER ════════════════════════════════════════════════════
w_head_txt   = widgets.Text(value="SYSTEM PROTOCOL // ANALYSIS", description="হেডার টেক্সট:", style=style, layout=layout_full)
w_head_sz    = widgets.IntSlider(value=20, min=10, max=60, description="সাইজ:", style=style)
w_head_col   = widgets.ColorPicker(value="#00E5FF", description="রঙ:", style=style)
w_head_y     = widgets.IntSlider(value=60, min=0, max=400, description="Y:", style=style)
w_head_align = widgets.ToggleButtons(options=["left","center","right"], value="left", description="Align:", style={"description_width":"80px","button_width":"80px"})
w_head_font  = widgets.Dropdown(options=["Regular","Medium","Bold"], value="Regular", description="Font Weight:", style=style)

# ══ 6. HERO TEXT (3 Lines) ═══════════════════════════════════════
w_hero1_txt = widgets.Text(value="কেন ছাত্রছাত্রীরা ভুল", description="Hero লাইন ১:", style=style, layout=layout_full)
w_hero2_txt = widgets.Text(value="পদ্ধতিতে পড়াশোনা করে?", description="Hero লাইন ২:", style=style, layout=layout_full)
w_hero3_txt = widgets.Text(value="", description="Hero লাইন ৩:", style=style, layout=layout_full)
w_show_hero3 = widgets.Checkbox(value=False, description="লাইন ৩ দেখান", style=style)

w_hero1_sz  = widgets.IntSlider(value=90, min=30, max=160, description="লাইন ১ সাইজ:", style=style)
w_hero2_sz  = widgets.IntSlider(value=90, min=30, max=160, description="লাইন ২ সাইজ:", style=style)
w_hero3_sz  = widgets.IntSlider(value=70, min=30, max=160, description="লাইন ৩ সাইজ:", style=style)

w_hero1_col = widgets.ColorPicker(value="#FFFFFF", description="লাইন ১ রঙ:", style=style)
w_hero2_col = widgets.ColorPicker(value="#00E5FF", description="লাইন ২ রঙ:", style=style)
w_hero3_col = widgets.ColorPicker(value="#B0BEC5", description="লাইন ৩ রঙ:", style=style)

w_hero1_y   = widgets.IntSlider(value=150, min=50, max=700, description="লাইন ১ Y:", style=style)
w_hero2_y   = widgets.IntSlider(value=270, min=50, max=700, description="লাইন ২ Y:", style=style)
w_hero3_y   = widgets.IntSlider(value=390, min=50, max=800, description="লাইন ৩ Y:", style=style)

w_hero_shadow = widgets.Checkbox(value=True, description="Text Shadow চালু", style=style)
w_hero_align  = widgets.ToggleButtons(options=["left","center","right"], value="left", description="Align:", style={"description_width":"80px","button_width":"80px"})
w_hero_font   = widgets.Dropdown(options=["Regular","Medium","Bold"], value="Bold", description="Font Weight:", style=style)

print("✅ Step 4: UI Controls (Canvas, Effects & Header) Ready!")

✅ Step 4: UI Controls (Canvas, Effects & Header) Ready!


In [5]:
# @title 🎛️ Step 5: UI Controls – Module, Content, QR & Export

# ══ 7. MODULE BOX (V4 Multi-Module & Glassmorphism) ═════════════════
w_show_mod       = widgets.Checkbox(value=True, description="মডিউল বক্স দেখান", style=style)
w_module_count   = widgets.IntSlider(value=1, min=1, max=3, description="মডিউল সংখ্যা:", style=style)
w_use_glass      = widgets.Checkbox(value=False, description="Glassmorphism চালু", style=style)
w_glass_opacity  = widgets.FloatSlider(value=0.3, min=0.1, max=0.9, step=0.05, description="Glass Opacity:", style=style)

w_mod_y          = widgets.IntSlider(value=450, min=200, max=1200, description="বক্স Y:", style=style, layout=layout_full)
w_mod_h          = widgets.IntSlider(value=420, min=150, max=900, description="বক্স উচ্চতা:", style=style, layout=layout_full)
w_mod_bg         = widgets.ColorPicker(value="#121826", description="বক্স BG:", style=style)
w_mod_rad        = widgets.IntSlider(value=30, min=0, max=100, description="রাউন্ডনেস:", style=style)
w_show_mod_border= widgets.Checkbox(value=True, description="Accent Border", style=style)
w_mod_border_col = widgets.ColorPicker(value="#00E5FF", description="Border রঙ:", style=style)
w_mod_shadow     = widgets.Checkbox(value=True, description="Drop Shadow", style=style)

# Badge
w_bdg_num   = widgets.Text(value="01", description="ব্যাজ নম্বর:", style=style)
w_bdg_title = widgets.Text(value="রিটেনশন ফ্যাক্টর", description="মডিউল নাম:", style=style, layout=layout_full)
w_bdg_sz    = widgets.IntSlider(value=24, min=10, max=50, description="নম্বর সাইজ:", style=style)
w_bdg_col   = widgets.ColorPicker(value="#00E5FF", description="নম্বর রঙ:", style=style)
w_bdg_shape = widgets.ToggleButtons(options=["circle","square","diamond"], value="circle", description="Badge Shape:", style={"description_width":"120px","button_width":"90px"})

# ══ 8. CONTENT ═══════════════════════════════════════════════════
w_err_title = widgets.Text(value="DETECTED ERROR:", description="Error টাইটেল:", style=style)
w_err_txt   = widgets.Textarea(value="মনে মনে পড়া বা প্যাসিভ লার্নিং।", description="Error বিস্তারিত:", rows=2, style=style, layout=layout_full)
w_err_col   = widgets.ColorPicker(value="#FF5722", description="Error রঙ:", style=style)

w_opt_title = widgets.Text(value="OPTIMAL PATH:", description="Solution টাইটেল:", style=style)
w_opt_txt   = widgets.Textarea(value="অ্যাক্টিভ রিকল এবং স্পেসড রিপিটেশন।", description="Solution বিস্তারিত:", rows=2, style=style, layout=layout_full)
w_opt_col   = widgets.ColorPicker(value="#00C853", description="Solution রঙ:", style=style)
w_body_font = widgets.Dropdown(options=["Regular","Medium","Bold"], value="Medium", description="Body Font:", style=style)

# ══ 9. STATS BLOCK ═══════════════════════════════════════════════
w_show_stats  = widgets.Checkbox(value=False, description="Stats Block দেখান", style=style)
w_stats_y     = widgets.IntSlider(value=1050, min=500, max=1800, description="Stats Y:", style=style, layout=layout_full)
w_stats_col   = widgets.ColorPicker(value="#00E5FF", description="সংখ্যার রঙ:", style=style)
w_stats_num1  = widgets.Text(value="৯৫%",    description="সংখ্যা ১:", style=style)
w_stats_lbl1  = widgets.Text(value="সাফল্যের হার",description="লেবেল ১:", style=style)
w_stats_num2  = widgets.Text(value="৪৮H",    description="সংখ্যা ২:", style=style)
w_stats_lbl2  = widgets.Text(value="কোর্স সময়",  description="লেবেল ২:", style=style)
w_stats_num3  = widgets.Text(value="১০K+",   description="সংখ্যা ৩:", style=style)
w_stats_lbl3  = widgets.Text(value="শিক্ষার্থী",  description="লেবেল ৩:", style=style)

# ══ 10. QR CODE (V4 Feature) ═════════════════════════════════════
w_show_qr = widgets.Checkbox(value=False, description="QR Code দেখান", style=style)
w_qr_url  = widgets.Text(value="https://yourwebsite.com", description="QR URL:", style=style, layout=layout_full)
w_qr_size = widgets.IntSlider(value=150, min=80, max=300, description="QR Size:", style=style)
w_qr_col  = widgets.ColorPicker(value="#FFFFFF", description="QR Color:", style=style)

# ══ 11. LOGO / WATERMARK ════════════════════════════════════════
w_show_logo     = widgets.Checkbox(value=False, description="Logo দেখান", style=style)
w_logo_upload   = widgets.FileUpload(accept="image/*", multiple=False, description="Logo Upload:", layout=widgets.Layout(width="300px"))
w_logo_size     = widgets.IntSlider(value=100, min=30, max=400, description="Logo সাইজ:", style=style)
w_logo_pos      = widgets.ToggleButtons(options=["top-left","top-right","bottom-left","bottom-right"], value="top-right", description="Logo অবস্থান:", style={"description_width":"120px","button_width":"110px"})
w_logo_opacity  = widgets.FloatSlider(value=1.0, min=0.1, max=1.0, step=0.05, description="Logo Opacity:", style=style)

# ══ 12. FOOTER ═══════════════════════════════════════════════════
w_show_footer = widgets.Checkbox(value=True, description="Footer দেখান", style=style)
w_foot_txt    = widgets.Text(value="© Your Name | your.website.com", description="Footer টেক্সট:", style=style, layout=layout_full)
w_foot_col    = widgets.ColorPicker(value="#607D8B", description="Footer রঙ:", style=style)
w_foot_sz     = widgets.IntSlider(value=22, min=12, max=40, description="Footer সাইজ:", style=style)

# ══ 13. LAYOUT & DIVIDER ═════════════════════════════════════════
w_margin_x     = widgets.IntSlider(value=80, min=0, max=200, description="সাইড মার্জিন:", style=style)
w_spacing      = widgets.IntSlider(value=45, min=0, max=100, description="লাইন স্পেসিং:", style=style)
w_show_divider = widgets.Checkbox(value=True, description="Divider Line", style=style)
w_divider_col  = widgets.ColorPicker(value="#00E5FF", description="Divider রঙ:", style=style)
w_divider_style= widgets.ToggleButtons(options=["solid","dotted","double","gradient"], value="solid", description="Divider Style:", style={"description_width":"120px","button_width":"90px"})

# ══ 14. EXPORT ═══════════════════════════════════════════════════
w_export_fmt     = widgets.ToggleButtons(options=["PNG","JPEG","WEBP"], value="PNG", description="Format:", style={"description_width":"80px","button_width":"70px"})
w_export_quality = widgets.IntSlider(value=92, min=50, max=100, description="Quality:", style=style)
w_filename       = widgets.Text(value="my_template_v4", description="File Name:", style=style, layout=layout_full)

print("✅ Step 5: UI Controls (Module, Content, QR & Export) Ready!")

✅ Step 5: UI Controls (Module, Content, QR & Export) Ready!


In [6]:
# @title 🎨 Step 6: Preset Color Themes (22 Professional Themes)

PRESETS = {
    # --- Original 12 Themes ---
    "🌌 Dark Space":       {"bg_start":"#050814","bg_end":"#1A1030","head_col":"#00E5FF","hero1_col":"#FFFFFF","hero2_col":"#00E5FF","mod_bg":"#121826","mod_border_col":"#00E5FF","err_col":"#FF5722","opt_col":"#00C853","divider_col":"#00E5FF","foot_col":"#607D8B","accent_bar_col":"#00E5FF"},
    "🔥 Sunset Orange":    {"bg_start":"#1A0A00","bg_end":"#3D1200","head_col":"#FF6D00","hero1_col":"#FFFFFF","hero2_col":"#FF6D00","mod_bg":"#1E1000","mod_border_col":"#FF6D00","err_col":"#FF1744","opt_col":"#76FF03","divider_col":"#FF6D00","foot_col":"#795548","accent_bar_col":"#FF6D00"},
    "💜 Purple Haze":      {"bg_start":"#0D001A","bg_end":"#1A0030","head_col":"#EA80FC","hero1_col":"#FFFFFF","hero2_col":"#EA80FC","mod_bg":"#160020","mod_border_col":"#EA80FC","err_col":"#FF4081","opt_col":"#69F0AE","divider_col":"#EA80FC","foot_col":"#9E9E9E","accent_bar_col":"#EA80FC"},
    "🌿 Forest Green":     {"bg_start":"#001209","bg_end":"#002614","head_col":"#69F0AE","hero1_col":"#FFFFFF","hero2_col":"#69F0AE","mod_bg":"#001A0C","mod_border_col":"#69F0AE","err_col":"#FF5252","opt_col":"#CCFF90","divider_col":"#69F0AE","foot_col":"#546E7A","accent_bar_col":"#69F0AE"},
    "🌅 Golden Hour":      {"bg_start":"#1A1200","bg_end":"#2D1F00","head_col":"#FFD600","hero1_col":"#FFFFFF","hero2_col":"#FFD600","mod_bg":"#1A1300","mod_border_col":"#FFD600","err_col":"#FF6E40","opt_col":"#B9F6CA","divider_col":"#FFD600","foot_col":"#78909C","accent_bar_col":"#FFD600"},
    "🩺 Midnight Medical": {"bg_start":"#020B18","bg_end":"#061E36","head_col":"#40C4FF","hero1_col":"#E3F2FD","hero2_col":"#40C4FF","mod_bg":"#0A1929","mod_border_col":"#40C4FF","err_col":"#EF5350","opt_col":"#00E676","divider_col":"#1565C0","foot_col":"#546E7A","accent_bar_col":"#40C4FF"},
    "🖤 Carbon Black":     {"bg_start":"#0A0A0A","bg_end":"#1C1C1C","head_col":"#E0E0E0","hero1_col":"#FFFFFF","hero2_col":"#BDBDBD","mod_bg":"#141414","mod_border_col":"#424242","err_col":"#EF9A9A","opt_col":"#A5D6A7","divider_col":"#424242","foot_col":"#616161","accent_bar_col":"#757575"},
    "🌊 Deep Ocean":       {"bg_start":"#001428","bg_end":"#00264D","head_col":"#82B1FF","hero1_col":"#E8EAF6","hero2_col":"#82B1FF","mod_bg":"#001933","mod_border_col":"#448AFF","err_col":"#FF6090","opt_col":"#64FFDA","divider_col":"#1A237E","foot_col":"#5C6BC0","accent_bar_col":"#448AFF"},
    "🍒 Cherry Blossom":   {"bg_start":"#1A0010","bg_end":"#33001F","head_col":"#F48FB1","hero1_col":"#FCE4EC","hero2_col":"#F48FB1","mod_bg":"#200015","mod_border_col":"#F06292","err_col":"#FF5252","opt_col":"#80CBC4","divider_col":"#880E4F","foot_col":"#AD1457","accent_bar_col":"#F06292"},
    "🏆 Royal Gold":       {"bg_start":"#0D0900","bg_end":"#1F1400","head_col":"#FFC400","hero1_col":"#FFF8E1","hero2_col":"#FFD740","mod_bg":"#150D00","mod_border_col":"#FFC400","err_col":"#FF6F00","opt_col":"#CCFF90","divider_col":"#FF8F00","foot_col":"#8D6E63","accent_bar_col":"#FFC400"},
    "🧊 Arctic Frost":     {"bg_start":"#011627","bg_end":"#022B4A","head_col":"#B2EBF2","hero1_col":"#E0F7FA","hero2_col":"#80DEEA","mod_bg":"#01243E","mod_border_col":"#4DD0E1","err_col":"#FF8A65","opt_col":"#A5F3A5","divider_col":"#0097A7","foot_col":"#607D8B","accent_bar_col":"#4DD0E1"},
    "🔬 Neon Lab":         {"bg_start":"#030014","bg_end":"#0A0028","head_col":"#E040FB","hero1_col":"#EDE7F6","hero2_col":"#7C4DFF","mod_bg":"#07001E","mod_border_col":"#7C4DFF","err_col":"#FF4081","opt_col":"#69F0AE","divider_col":"#651FFF","foot_col":"#7E57C2","accent_bar_col":"#7C4DFF"},

    # --- New 10 Professional Themes ---
    "🤖 Cyberpunk Glow":   {"bg_start":"#0F0F1A","bg_end":"#1A0B2E","head_col":"#FAFF00","hero1_col":"#FFFFFF","hero2_col":"#00E5FF","mod_bg":"#140D26","mod_border_col":"#FAFF00","err_col":"#FF003C","opt_col":"#00FF9D","divider_col":"#FAFF00","foot_col":"#6E6E99","accent_bar_col":"#FAFF00"},
    "🤍 Minimalist Clean": {"bg_start":"#F8F9FA","bg_end":"#E9ECEF","head_col":"#212529","hero1_col":"#343A40","hero2_col":"#495057","mod_bg":"#FFFFFF","mod_border_col":"#DEE2E6","err_col":"#DC3545","opt_col":"#198754","divider_col":"#CED4DA","foot_col":"#6C757D","accent_bar_col":"#6C757D"},
    "💎 Emerald Elegance": {"bg_start":"#00170F","bg_end":"#002E1E","head_col":"#50C878","hero1_col":"#FFFFFF","hero2_col":"#50C878","mod_bg":"#002115","mod_border_col":"#50C878","err_col":"#FF4A4A","opt_col":"#98FF98","divider_col":"#50C878","foot_col":"#6B8E7B","accent_bar_col":"#50C878"},
    "🩸 Crimson Night":    {"bg_start":"#120000","bg_end":"#2B0000","head_col":"#FF1E1E","hero1_col":"#FFFFFF","hero2_col":"#FF1E1E","mod_bg":"#1C0000","mod_border_col":"#FF1E1E","err_col":"#FF6B6B","opt_col":"#4CAF50","divider_col":"#FF1E1E","foot_col":"#804040","accent_bar_col":"#FF1E1E"},
    "🍂 Autumn Warmth":    {"bg_start":"#2E1503","bg_end":"#4A2511","head_col":"#FFB04A","hero1_col":"#FFFFFF","hero2_col":"#FFB04A","mod_bg":"#3D1D0A","mod_border_col":"#FFB04A","err_col":"#FF5959","opt_col":"#A3FF85","divider_col":"#FFB04A","foot_col":"#967866","accent_bar_col":"#FFB04A"},
    "🔮 Lavender Dream":   {"bg_start":"#170A21","bg_end":"#2D1B3D","head_col":"#D4A1FF","hero1_col":"#FFFFFF","hero2_col":"#D4A1FF","mod_bg":"#241433","mod_border_col":"#D4A1FF","err_col":"#FF6B9E","opt_col":"#A1FFD4","divider_col":"#D4A1FF","foot_col":"#8B7B9E","accent_bar_col":"#D4A1FF"},
    "🏢 Corporate Navy":   {"bg_start":"#030C1A","bg_end":"#081E3D","head_col":"#64A8FF","hero1_col":"#FFFFFF","hero2_col":"#64A8FF","mod_bg":"#06152B","mod_border_col":"#64A8FF","err_col":"#FF5C5C","opt_col":"#5CFFB6","divider_col":"#64A8FF","foot_col":"#6B83A1","accent_bar_col":"#64A8FF"},
    "🌸 Rose Gold":        {"bg_start":"#1F1115","bg_end":"#361D24","head_col":"#F4C4B7","hero1_col":"#FFFFFF","hero2_col":"#F4C4B7","mod_bg":"#2B181E","mod_border_col":"#F4C4B7","err_col":"#FF7A7A","opt_col":"#B7F4D8","divider_col":"#F4C4B7","foot_col":"#A18C91","accent_bar_col":"#F4C4B7"},
    "🍫 Mint Chocolate":   {"bg_start":"#1A110C","bg_end":"#2E1F16","head_col":"#98FFB3","hero1_col":"#FFFFFF","hero2_col":"#98FFB3","mod_bg":"#241710","mod_border_col":"#98FFB3","err_col":"#FF6B6B","opt_col":"#D4FF98","divider_col":"#98FFB3","foot_col":"#8C817A","accent_bar_col":"#98FFB3"},
    "⚙️ Slate & Copper":   {"bg_start":"#12161A","bg_end":"#1F262E","head_col":"#E68A5C","hero1_col":"#FFFFFF","hero2_col":"#E68A5C","mod_bg":"#181D24","mod_border_col":"#E68A5C","err_col":"#FF5C5C","opt_col":"#5CE6A1","divider_col":"#E68A5C","foot_col":"#7A848F","accent_bar_col":"#E68A5C"},
}

preset_buttons = []
out_preset = widgets.Output()

def _mk_preset_handler(name, vals):
    def _h(b):
        _map = {
            "bg_start":      w_bg_start,
            "bg_end":        w_bg_end,
            "head_col":      w_head_col,
            "hero1_col":     w_hero1_col,
            "hero2_col":     w_hero2_col,
            "mod_bg":        w_mod_bg,
            "mod_border_col":w_mod_border_col,
            "err_col":       w_err_col,
            "opt_col":       w_opt_col,
            "divider_col":   w_divider_col,
            "foot_col":      w_foot_col,
            "accent_bar_col":w_accent_bar_col,
        }
        for k, wgt in _map.items():
            if k in vals:
                wgt.value = vals[k]
        with out_preset:
            clear_output()
            print(f"✅ Theme Applied: {name}")
    return _h

for name, vals in PRESETS.items():
    btn = widgets.Button(description=name,
                          layout=widgets.Layout(width="195px", height="38px"),
                          style={"button_color":"#1E293B"})
    btn.on_click(_mk_preset_handler(name, vals))
    preset_buttons.append(btn)

_rows = [preset_buttons[i:i+4] for i in range(0, len(preset_buttons), 4)]
preset_box = widgets.VBox([
    widgets.HTML("<b style='font-size:14px;'>🎨 Color Themes (22)</b>"),
    *[widgets.HBox(row) for row in _rows],
    out_preset
])
print(f"✅ Step 6: {len(PRESETS)} Themes Ready!")

✅ Step 6: 22 Themes Ready!


In [7]:
# @title 🎨 Step 6b: Additional 10 Professional Color Themes

NEW_PRESETS = {
    "🎨 Pastel Dreams":     {"bg_start":"#2B1B3A","bg_end":"#1A1125","head_col":"#FFB6C1","hero1_col":"#FFFFFF","hero2_col":"#E6E6FA","mod_bg":"#2D1F3A","mod_border_col":"#FFB6C1","err_col":"#FF69B4","opt_col":"#98FB98","divider_col":"#DDA0DD","foot_col":"#B0C4DE","accent_bar_col":"#FFB6C1"},
    "🎪 Vintage Paper":     {"bg_start":"#8B7E6B","bg_end":"#5D4F3F","head_col":"#F4E4C1","hero1_col":"#FFF9E6","hero2_col":"#D4B48C","mod_bg":"#6B5A48","mod_border_col":"#C9A87C","err_col":"#B22222","opt_col":"#556B2F","divider_col":"#A67B5B","foot_col":"#8B7355","accent_bar_col":"#D4B48C"},
    "🌅 Solar Flare":       {"bg_start":"#4A0D0D","bg_end":"#B23B0A","head_col":"#FFD700","hero1_col":"#FFFFFF","hero2_col":"#FFA500","mod_bg":"#6B1E0A","mod_border_col":"#FF8C00","err_col":"#FF4500","opt_col":"#32CD32","divider_col":"#FFA500","foot_col":"#CD853F","accent_bar_col":"#FF8C00"},
    "🍇 Eggplant":          {"bg_start":"#1E0F1A","bg_end":"#2D1B2A","head_col":"#DDA0DD","hero1_col":"#FFFFFF","hero2_col":"#BA55D3","mod_bg":"#231323","mod_border_col":"#9932CC","err_col":"#DB7093","opt_col":"#7B68EE","divider_col":"#9370DB","foot_col":"#8A6D8A","accent_bar_col":"#BA55D3"},
    "🌲 Evergreen":         {"bg_start":"#0A1F0E","bg_end":"#1A3A1E","head_col":"#90EE90","hero1_col":"#FFFFFF","hero2_col":"#32CD32","mod_bg":"#0F2A13","mod_border_col":"#228B22","err_col":"#FF6347","opt_col":"#ADFF2F","divider_col":"#3CB371","foot_col":"#548B54","accent_bar_col":"#32CD32"},
    "🌃 Tokyo Night":       {"bg_start":"#0A0F1E","bg_end":"#1A1F35","head_col":"#7B68EE","hero1_col":"#FFFFFF","hero2_col":"#6A5ACD","mod_bg":"#121827","mod_border_col":"#483D8B","err_col":"#FF6B6B","opt_col":"#4ECDC4","divider_col":"#7B68EE","foot_col":"#6B8E9B","accent_bar_col":"#6A5ACD"},
    "🎭 Royal Crimson":     {"bg_start":"#1A0B0F","bg_end":"#2D1219","head_col":"#DC143C","hero1_col":"#FFFFFF","hero2_col":"#B22222","mod_bg":"#231014","mod_border_col":"#8B0000","err_col":"#FFA07A","opt_col":"#FFD700","divider_col":"#CD5C5C","foot_col":"#A0522D","accent_bar_col":"#B22222"},
    "🌿 Sage & Olive":      {"bg_start":"#1A241A","bg_end":"#2A342A","head_col":"#9ACD32","hero1_col":"#FFFFFF","hero2_col":"#6B8E23","mod_bg":"#1F2A1F","mod_border_col":"#556B2F","err_col":"#CD853F","opt_col":"#BDB76B","divider_col":"#8FBC8F","foot_col":"#6B8E4F","accent_bar_col":"#6B8E23"},
    "💎 Sapphire":          {"bg_start":"#001122","bg_end":"#002244","head_col":"#4169E1","hero1_col":"#FFFFFF","hero2_col":"#1E90FF","mod_bg":"#001933","mod_border_col":"#0F52BA","err_col":"#FF7F50","opt_col":"#00FA9A","divider_col":"#4682B4","foot_col":"#36648B","accent_bar_col":"#1E90FF"},
    "🪨 Stone & Clay":      {"bg_start":"#2C2418","bg_end":"#3E3428","head_col":"#E6A56F","hero1_col":"#FFFFFF","hero2_col":"#C19A6B","mod_bg":"#322A1F","mod_border_col":"#B78C5A","err_col":"#C04040","opt_col":"#7CB490","divider_col":"#A67B5B","foot_col":"#8B7355","accent_bar_col":"#C19A6B"},
}

# Add new themes to existing PRESETS
PRESETS.update(NEW_PRESETS)

# Recreate preset buttons with updated themes
preset_buttons = []
out_preset = widgets.Output()

def _mk_preset_handler(name, vals):
    def _h(b):
        _map = {
            "bg_start":      w_bg_start,
            "bg_end":        w_bg_end,
            "head_col":      w_head_col,
            "hero1_col":     w_hero1_col,
            "hero2_col":     w_hero2_col,
            "mod_bg":        w_mod_bg,
            "mod_border_col":w_mod_border_col,
            "err_col":       w_err_col,
            "opt_col":       w_opt_col,
            "divider_col":   w_divider_col,
            "foot_col":      w_foot_col,
            "accent_bar_col":w_accent_bar_col,
        }
        for k, wgt in _map.items():
            if k in vals:
                wgt.value = vals[k]
        with out_preset:
            clear_output()
            print(f"✅ Theme Applied: {name}")
    return _h

for name, vals in PRESETS.items():
    btn = widgets.Button(description=name,
                          layout=widgets.Layout(width="195px", height="38px"),
                          style={"button_color":"#1E293B"})
    btn.on_click(_mk_preset_handler(name, vals))
    preset_buttons.append(btn)

_rows = [preset_buttons[i:i+4] for i in range(0, len(preset_buttons), 4)]
preset_box = widgets.VBox([
    widgets.HTML("<b style='font-size:14px;'>🎨 Color Themes (32 Total)</b>"),
    *[widgets.HBox(row) for row in _rows],
    out_preset
])

print(f"✅ Added 10 new themes! Total: {len(PRESETS)} themes")

✅ Added 10 new themes! Total: 32 themes


In [8]:
# @title 🎨 Step 6c: 10 Full Professional Template Styles

STYLE_PRESETS = {
    "🚀 Modern Tech": {
        "cv_w": 1080, "cv_h": 1350,
        "bg_start": "#0A0F1F", "bg_end": "#1A1F35", "grad_dir": "vertical",
        "show_accent_bar": True, "accent_bar_h": 8, "accent_bar_col": "#00E5FF",
        "head_txt": "TECH INSIGHTS // 2025", "head_sz": 22, "head_col": "#00E5FF", "head_y": 60, "head_align": "left", "head_font": "Bold",
        "hero1_txt": "Build Scalable", "hero1_sz": 100, "hero1_col": "#FFFFFF", "hero1_y": 140,
        "hero2_txt": "AI Systems", "hero2_sz": 100, "hero2_col": "#00E5FF", "hero2_y": 260,
        "show_hero3": True, "hero3_txt": "with confidence", "hero3_sz": 70, "hero3_col": "#B0BEC5", "hero3_y": 380,
        "hero_shadow": True, "hero_align": "left", "hero_font": "Bold",
        "show_mod": True, "module_count": 1, "use_glass": True, "glass_opacity": 0.3,
        "mod_y": 480, "mod_h": 400, "mod_bg": "#121826", "mod_rad": 30,
        "show_mod_border": True, "mod_border_col": "#00E5FF", "mod_shadow": True,
        "bdg_num": "01", "bdg_title": "KEY PRINCIPLE", "bdg_col": "#00E5FF", "bdg_shape": "circle",
        "err_title": "CHALLENGE:", "err_txt": "Data quality & model drift", "err_col": "#FF5722",
        "opt_title": "SOLUTION:", "opt_txt": "Automated retraining pipelines", "opt_col": "#00C853",
        "body_font": "Medium",
        "show_stats": True, "stats_y": 950, "stats_col": "#00E5FF",
        "stats_num1": "99.9%", "stats_lbl1": "Uptime", "stats_num2": "<5ms", "stats_lbl2": "Latency", "stats_num3": "10M+", "stats_lbl3": "Requests",
        "show_qr": True, "qr_url": "https://techdocs.example", "qr_size": 140, "qr_col": "#FFFFFF",
        "show_logo": True, "logo_size": 100, "logo_pos": "top-right", "logo_opacity": 0.9,
        "show_footer": True, "foot_txt": "© TechCorp | docs.techcorp.com", "foot_col": "#607D8B", "foot_sz": 22,
        "margin_x": 80, "spacing": 45,
        "show_divider": True, "divider_col": "#00E5FF", "divider_style": "solid",
    },
    "💼 Corporate Clean": {
        "cv_w": 1200, "cv_h": 628,
        "bg_start": "#F8F9FA", "bg_end": "#E9ECEF", "grad_dir": "vertical",
        "show_accent_bar": False,
        "head_txt": "QUARTERLY REPORT", "head_sz": 18, "head_col": "#495057", "head_y": 40, "head_align": "left", "head_font": "Medium",
        "hero1_txt": "Revenue Growth", "hero1_sz": 72, "hero1_col": "#212529", "hero1_y": 100,
        "hero2_txt": "+23% YoY", "hero2_sz": 72, "hero2_col": "#0CA678", "hero2_y": 180,
        "show_hero3": False,
        "hero_shadow": False, "hero_align": "left", "hero_font": "Bold",
        "show_mod": False,
        "show_stats": True, "stats_y": 320, "stats_col": "#0CA678",
        "stats_num1": "$4.2M", "stats_lbl1": "Q2 Revenue", "stats_num2": "156", "stats_lbl2": "New Clients", "stats_num3": "94%", "stats_lbl3": "Retention",
        "show_qr": False,
        "show_logo": True, "logo_size": 70, "logo_pos": "top-left", "logo_opacity": 1.0,
        "show_footer": True, "foot_txt": "confidential – for internal use", "foot_col": "#868E96", "foot_sz": 16,
        "margin_x": 60, "spacing": 30,
        "show_divider": True, "divider_col": "#CED4DA", "divider_style": "solid",
    },
    "📱 Social Media Promo": {
        "cv_w": 1080, "cv_h": 1920,
        "bg_start": "#6A1B9A", "bg_end": "#4A148C", "grad_dir": "vertical",
        "show_accent_bar": True, "accent_bar_h": 12, "accent_bar_col": "#FFD600",
        "head_txt": "🔥 LIMITED OFFER", "head_sz": 28, "head_col": "#FFD600", "head_y": 80, "head_align": "center", "head_font": "Bold",
        "hero1_txt": "50% OFF", "hero1_sz": 140, "hero1_col": "#FFFFFF", "hero1_y": 200,
        "hero2_txt": "First Month", "hero2_sz": 90, "hero2_col": "#FFD600", "hero2_y": 360,
        "show_hero3": True, "hero3_txt": "for new subscribers", "hero3_sz": 60, "hero3_col": "#E1BEE7", "hero3_y": 480,
        "hero_shadow": True, "hero_align": "center", "hero_font": "Bold",
        "show_mod": True, "module_count": 2, "use_glass": True, "glass_opacity": 0.25,
        "mod_y": 600, "mod_h": 300, "mod_bg": "#4A148C", "mod_rad": 40,
        "show_mod_border": True, "mod_border_col": "#FFD600", "mod_shadow": True,
        "bdg_num": "⭐", "bdg_title": "FEATURE 1", "bdg_col": "#FFD600", "bdg_shape": "square",
        "err_title": "DON'T MISS:", "err_txt": "Early access bonus", "err_col": "#FFB74D",
        "opt_title": "INCLUDES:", "opt_txt": "All premium tools + support", "opt_col": "#81C784",
        "body_font": "Medium",
        "show_stats": False,
        "show_qr": True, "qr_url": "https://claim.offer", "qr_size": 200, "qr_col": "#FFFFFF",
        "show_logo": True, "logo_size": 120, "logo_pos": "bottom-left", "logo_opacity": 0.8,
        "show_footer": True, "foot_txt": "@yourbrand | linkin.bio/offer", "foot_col": "#CE93D8", "foot_sz": 26,
        "margin_x": 60, "spacing": 35,
        "show_divider": True, "divider_col": "#FFD600", "divider_style": "dotted",
    },
    "📚 Educational": {
        "cv_w": 1080, "cv_h": 1350,
        "bg_start": "#0D2B45", "bg_end": "#144A6F", "grad_dir": "vertical",
        "show_accent_bar": True, "accent_bar_h": 6, "accent_bar_col": "#FFB74D",
        "head_txt": "LEARNING LAB // SCIENCE", "head_sz": 22, "head_col": "#FFB74D", "head_y": 50, "head_align": "left", "head_font": "Medium",
        "hero1_txt": "How Memory", "hero1_sz": 90, "hero1_col": "#FFFFFF", "hero1_y": 130,
        "hero2_txt": "Actually Works", "hero2_sz": 90, "hero2_col": "#FFB74D", "hero2_y": 240,
        "show_hero3": True, "hero3_txt": "(neuroscience explained)", "hero3_sz": 50, "hero3_col": "#B0C4DE", "hero3_y": 350,
        "hero_shadow": True, "hero_align": "left", "hero_font": "Bold",
        "show_mod": True, "module_count": 2, "use_glass": False,
        "mod_y": 450, "mod_h": 320, "mod_bg": "#1E3A5F", "mod_rad": 20,
        "show_mod_border": True, "mod_border_col": "#FFB74D", "mod_shadow": True,
        "bdg_num": "01", "bdg_title": "ENCODING", "bdg_col": "#FFB74D", "bdg_shape": "circle",
        "err_title": "COMMON MYTH:", "err_txt": "Learning styles (visual/audio) are effective", "err_col": "#FF8A80",
        "opt_title": "TRUTH:", "opt_txt": "Spaced repetition & active recall", "opt_col": "#B9F6CA",
        "body_font": "Medium",
        "show_stats": True, "stats_y": 1100, "stats_col": "#FFB74D",
        "stats_num1": "50%", "stats_lbl1": "Faster retention", "stats_num2": "2x", "stats_lbl2": "Long-term recall", "stats_num3": "10min", "stats_lbl3": "Daily practice",
        "show_qr": True, "qr_url": "https://course.link/memory", "qr_size": 130, "qr_col": "#FFFFFF",
        "show_logo": True, "logo_size": 90, "logo_pos": "top-right", "logo_opacity": 0.8,
        "show_footer": True, "foot_txt": "© EduLab | edulab.org/memory", "foot_col": "#8CA6C9", "foot_sz": 22,
        "margin_x": 70, "spacing": 40,
        "show_divider": True, "divider_col": "#FFB74D", "divider_style": "solid",
    },
    "🎉 Event Announcement": {
        "cv_w": 1200, "cv_h": 1200,
        "bg_start": "#880E4F", "bg_end": "#4A0072", "grad_dir": "diagonal",
        "show_accent_bar": True, "accent_bar_h": 12, "accent_bar_col": "#FFEA00",
        "head_txt": "🎤 LIVE WEBINAR", "head_sz": 32, "head_col": "#FFEA00", "head_y": 70, "head_align": "center", "head_font": "Bold",
        "hero1_txt": "Future of", "hero1_sz": 100, "hero1_col": "#FFFFFF", "hero1_y": 180,
        "hero2_txt": "AI in 2025", "hero2_sz": 110, "hero2_col": "#FFEA00", "hero2_y": 300,
        "show_hero3": True, "hero3_txt": "with Dr. Sarah Chen", "hero3_sz": 50, "hero3_col": "#F8BBD0", "hero3_y": 430,
        "hero_shadow": True, "hero_align": "center", "hero_font": "Bold",
        "show_mod": False,
        "show_stats": False,
        "show_qr": True, "qr_url": "https://event.reg/ai2025", "qr_size": 180, "qr_col": "#FFEA00",
        "show_logo": True, "logo_size": 120, "logo_pos": "bottom-right", "logo_opacity": 0.9,
        "show_footer": True, "foot_txt": "May 15, 2025 · 3 PM GMT | register now", "foot_col": "#FCE4EC", "foot_sz": 26,
        "margin_x": 80, "spacing": 0,
        "show_divider": True, "divider_col": "#FFEA00", "divider_style": "double",
    },
    "📦 Product Launch": {
        "cv_w": 1080, "cv_h": 1080,
        "bg_start": "#263238", "bg_end": "#37474F", "grad_dir": "horizontal",
        "show_accent_bar": True, "accent_bar_h": 8, "accent_bar_col": "#64B5F6",
        "head_txt": "INTRODUCING", "head_sz": 30, "head_col": "#64B5F6", "head_y": 60, "head_align": "left", "head_font": "Medium",
        "hero1_txt": "NOVA X", "hero1_sz": 140, "hero1_col": "#FFFFFF", "hero1_y": 150,
        "hero2_txt": "Pro Edition", "hero2_sz": 80, "hero2_col": "#64B5F6", "hero2_y": 300,
        "show_hero3": True, "hero3_txt": "The ultimate smart hub", "hero3_sz": 50, "hero3_col": "#B0BEC5", "hero3_y": 390,
        "hero_shadow": True, "hero_align": "left", "hero_font": "Bold",
        "show_mod": True, "module_count": 1, "use_glass": True, "glass_opacity": 0.2,
        "mod_y": 500, "mod_h": 350, "mod_bg": "#2C3E50", "mod_rad": 40,
        "show_mod_border": True, "mod_border_col": "#64B5F6", "mod_shadow": True,
        "bdg_num": "✨", "bdg_title": "KEY FEATURES", "bdg_col": "#64B5F6", "bdg_shape": "diamond",
        "err_title": "OLD:", "err_txt": "Limited connectivity", "err_col": "#EF5350",
        "opt_title": "NEW:", "opt_txt": "AI-powered mesh network", "opt_col": "#66BB6A",
        "body_font": "Medium",
        "show_stats": True, "stats_y": 920, "stats_col": "#64B5F6",
        "stats_num1": "2x", "stats_lbl1": "Faster", "stats_num2": "50%", "stats_lbl2": "More range", "stats_num3": "24/7", "stats_lbl3": "Support",
        "show_qr": True, "qr_url": "https://product.site/nova", "qr_size": 130, "qr_col": "#FFFFFF",
        "show_logo": True, "logo_size": 90, "logo_pos": "top-right", "logo_opacity": 1.0,
        "show_footer": True, "foot_txt": "pre-order now | nova.tech", "foot_col": "#90A4AE", "foot_sz": 22,
        "margin_x": 70, "spacing": 40,
        "show_divider": True, "divider_col": "#64B5F6", "divider_style": "gradient",
    },
    "🎨 Portfolio Highlight": {
        "cv_w": 1200, "cv_h": 1500,
        "bg_start": "#1E1E2E", "bg_end": "#2D2D44", "grad_dir": "vertical",
        "show_accent_bar": True, "accent_bar_h": 6, "accent_bar_col": "#F5C542",
        "head_txt": "CREATIVE PORTFOLIO", "head_sz": 24, "head_col": "#F5C542", "head_y": 50, "head_align": "right", "head_font": "Medium",
        "hero1_txt": "Alex Rivera", "hero1_sz": 110, "hero1_col": "#FFFFFF", "hero1_y": 140,
        "hero2_txt": "Visual Designer", "hero2_sz": 70, "hero2_col": "#F5C542", "hero2_y": 270,
        "show_hero3": True, "hero3_txt": "specializing in brand identity", "hero3_sz": 45, "hero3_col": "#C0C0C0", "hero3_y": 350,
        "hero_shadow": True, "hero_align": "right", "hero_font": "Bold",
        "show_mod": False,
        "show_stats": True, "stats_y": 500, "stats_col": "#F5C542",
        "stats_num1": "50+", "stats_lbl1": "Projects", "stats_num2": "8", "stats_lbl2": "Awards", "stats_num3": "12", "stats_lbl3": "Years exp",
        "show_qr": False,
        "show_logo": True, "logo_size": 100, "logo_pos": "bottom-left", "logo_opacity": 0.7,
        "show_footer": True, "foot_txt": "alex.design | @alex_rivera", "foot_col": "#9E9E9E", "foot_sz": 22,
        "margin_x": 80, "spacing": 30,
        "show_divider": True, "divider_col": "#F5C542", "divider_style": "solid",
    },
    "📈 Webinar / Workshop": {
        "cv_w": 1280, "cv_h": 720,
        "bg_start": "#0A1929", "bg_end": "#1A2A3A", "grad_dir": "vertical",
        "show_accent_bar": True, "accent_bar_h": 8, "accent_bar_col": "#66BB6A",
        "head_txt": "WORKSHOP SERIES", "head_sz": 26, "head_col": "#66BB6A", "head_y": 50, "head_align": "left", "head_font": "Bold",
        "hero1_txt": "Data Science", "hero1_sz": 90, "hero1_col": "#FFFFFF", "hero1_y": 130,
        "hero2_txt": "with Python", "hero2_sz": 70, "hero2_col": "#66BB6A", "hero2_y": 240,
        "show_hero3": True, "hero3_txt": "intermediate level", "hero3_sz": 40, "hero3_col": "#B0BEC5", "hero3_y": 320,
        "hero_shadow": True, "hero_align": "left", "hero_font": "Bold",
        "show_mod": True, "module_count": 2, "use_glass": False,
        "mod_y": 380, "mod_h": 200, "mod_bg": "#1E2A3A", "mod_rad": 15,
        "show_mod_border": True, "mod_border_col": "#66BB6A", "mod_shadow": True,
        "bdg_num": "1", "bdg_title": "MODULE 1", "bdg_col": "#66BB6A", "bdg_shape": "circle",
        "err_title": "PREREQ:", "err_txt": "Basic Python knowledge", "err_col": "#FFA726",
        "opt_title": "TOPICS:", "opt_txt": "Pandas, NumPy, Visualization", "opt_col": "#66BB6A",
        "body_font": "Medium",
        "show_stats": False,
        "show_qr": True, "qr_url": "https://workshop.link/register", "qr_size": 120, "qr_col": "#FFFFFF",
        "show_logo": True, "logo_size": 80, "logo_pos": "top-right", "logo_opacity": 0.9,
        "show_footer": True, "foot_txt": "June 10-12 · 10 AM EST | register now", "foot_col": "#78909C", "foot_sz": 20,
        "margin_x": 60, "spacing": 35,
        "show_divider": True, "divider_col": "#66BB6A", "divider_style": "solid",
    },
    "📊 Infographic Style": {
        "cv_w": 1080, "cv_h": 1350,
        "bg_start": "#F1F8E9", "bg_end": "#DCEDC8", "grad_dir": "vertical",
        "show_accent_bar": True, "accent_bar_h": 12, "accent_bar_col": "#2E7D32",
        "head_txt": "ENVIRONMENTAL IMPACT", "head_sz": 28, "head_col": "#1B5E20", "head_y": 50, "head_align": "center", "head_font": "Bold",
        "hero1_txt": "Plastic Waste", "hero1_sz": 100, "hero1_col": "#1B5E20", "hero1_y": 130,
        "hero2_txt": "by the Numbers", "hero2_sz": 70, "hero2_col": "#2E7D32", "hero2_y": 250,
        "show_hero3": False,
        "hero_shadow": False, "hero_align": "center", "hero_font": "Bold",
        "show_mod": True, "module_count": 3, "use_glass": False,
        "mod_y": 350, "mod_h": 200, "mod_bg": "#FFFFFF", "mod_rad": 20,
        "show_mod_border": True, "mod_border_col": "#2E7D32", "mod_shadow": True,
        "bdg_num": "01", "bdg_title": "Global", "bdg_col": "#2E7D32", "bdg_shape": "square",
        "err_title": "PROBLEM:", "err_txt": "8M tons enter oceans yearly", "err_col": "#C62828",
        "opt_title": "SOLUTION:", "opt_txt": "Reduce, reuse, recycle", "opt_col": "#2E7D32",
        "body_font": "Regular",
        "show_stats": True, "stats_y": 1150, "stats_col": "#1B5E20",
        "stats_num1": "50%", "stats_lbl1": "Single-use", "stats_num2": "20%", "stats_lbl2": "Recycled", "stats_num3": "2050", "stats_lbl3": "More than fish",
        "show_qr": True, "qr_url": "https://green.org/act", "qr_size": 140, "qr_col": "#1B5E20",
        "show_logo": False,
        "show_footer": True, "foot_txt": "source: UN Environment 2024", "foot_col": "#558B2F", "foot_sz": 20,
        "margin_x": 60, "spacing": 30,
        "show_divider": True, "divider_col": "#2E7D32", "divider_style": "dotted",
    },
    "💬 Minimalist Quote": {
        "cv_w": 1080, "cv_h": 1080,
        "bg_start": "#F5F5F5", "bg_end": "#EEEEEE", "grad_dir": "vertical",
        "show_accent_bar": False,
        "head_txt": "", "head_sz": 0,
        "hero1_txt": "“Simplicity", "hero1_sz": 100, "hero1_col": "#212121", "hero1_y": 300,
        "hero2_txt": "is the ultimate", "hero2_sz": 90, "hero2_col": "#757575", "hero2_y": 420,
        "show_hero3": True, "hero3_txt": "sophistication.”", "hero3_sz": 100, "hero3_col": "#212121", "hero3_y": 530,
        "hero_shadow": False, "hero_align": "center", "hero_font": "Medium",
        "show_mod": False,
        "show_stats": False,
        "show_qr": False,
        "show_logo": True, "logo_size": 60, "logo_pos": "bottom-right", "logo_opacity": 0.5,
        "show_footer": True, "foot_txt": "— Leonardo da Vinci", "foot_col": "#9E9E9E", "foot_sz": 30,
        "margin_x": 80, "spacing": 0,
        "show_divider": False,
    },
}

style_preset_buttons = []
out_style_preset = widgets.Output()

def _mk_style_preset_handler(name, vals):
    def _h(b):
        # Save current state for undo
        _save_snapshot()
        # Apply all values
        for k, v in vals.items():
            w = widget_mapping.get(k)
            if w is not None:
                try:
                    w.value = v
                except Exception:
                    pass
        with out_style_preset:
            clear_output()
            print(f"✅ Style Preset Applied: {name}")
    return _h

for name, vals in STYLE_PRESETS.items():
    btn = widgets.Button(description=name,
                          layout=widgets.Layout(width="210px", height="42px"),
                          style={"button_color": "#1E3A5F", "font_weight": "bold"})
    btn.on_click(_mk_style_preset_handler(name, vals))
    style_preset_buttons.append(btn)

# Arrange style preset buttons in rows of 3
style_rows = [style_preset_buttons[i:i+3] for i in range(0, len(style_preset_buttons), 3)]
style_preset_box = widgets.VBox([
    widgets.HTML("<b style='font-size:16px; color:#00E5FF;'>📋 Full Template Styles (10 Professional Layouts)</b>"),
    *[widgets.HBox(row, layout=widgets.Layout(justify_content="center", gap="8px")) for row in style_rows],
    out_style_preset
])

print(f"✅ Added {len(STYLE_PRESETS)} full template styles!")

✅ Added 10 full template styles!


In [9]:
# @title 📑 Step 7: Layout – Tabs Organization

# Tab 1 – Canvas
tab1 = widgets.VBox([
    widgets.HTML("<b>📐 Canvas Size & Gradient</b>"),
    w_cv_w, w_cv_h,
    widgets.HBox([w_bg_start, w_bg_end]),
    w_grad_dir,
    widgets.HTML("<hr><b>📱 Aspect Ratio Presets (এক ক্লিকে)</b>"),
    widgets.HBox(_aspect_btns[:3]),
    widgets.HBox(_aspect_btns[3:]),
])

# Tab 2 – Background Effects
tab2 = widgets.VBox([
    widgets.HTML("<b>🎨 Top Accent Bar</b>"),
    w_show_accent_bar, widgets.HBox([w_accent_bar_h, w_accent_bar_col]),
    widgets.HTML("<hr><b>🖼️ Background Image Upload</b>"),
    w_bg_img_upload, w_bg_img_opacity,
    widgets.HTML("<hr><b>🔲 Pattern Overlay</b>"),
    w_show_pattern, w_pattern_type, widgets.HBox([w_pattern_col, w_pattern_opacity]), w_pattern_spacing,
    widgets.HTML("<hr><b>🌫️ Advanced Effects (V4)</b>"),
    w_use_noise, w_noise_opacity,
])

# Tab 3 – Header & Hero
tab3 = widgets.VBox([
    widgets.HTML("<b>📋 Header</b>"),
    w_head_txt, widgets.HBox([w_head_sz, w_head_col]), widgets.HBox([w_head_y, w_head_font]), w_head_align,
    widgets.HTML("<hr><b>🎯 Hero Text (৩ লাইন)</b>"),
    w_hero1_txt, w_hero2_txt, w_show_hero3, w_hero3_txt,
    widgets.HBox([w_hero1_sz, w_hero2_sz, w_hero3_sz]),
    widgets.HBox([w_hero1_col, w_hero2_col, w_hero3_col]),
    widgets.HBox([w_hero1_y, w_hero2_y, w_hero3_y]),
    w_hero_shadow, w_hero_align, w_hero_font,
])

# Tab 4 – Module Box & Badge
tab4 = widgets.VBox([
    widgets.HTML("<b>📦 Module Box (V4: Multi-Module & Glassmorphism)</b>"),
    w_show_mod, w_module_count,
    widgets.HBox([w_use_glass, w_glass_opacity]),
    widgets.HBox([w_mod_y, w_mod_h]), widgets.HBox([w_mod_bg, w_mod_rad]),
    widgets.HBox([w_show_mod_border, w_mod_border_col]), w_mod_shadow,
    widgets.HTML("<hr><b>🏷️ Badge</b>"),
    widgets.HBox([w_bdg_num, w_bdg_title]), widgets.HBox([w_bdg_sz, w_bdg_col]), w_bdg_shape,
])

# Tab 5 – Content & Stats
tab5 = widgets.VBox([
    widgets.HTML("<b>❌ Error Section</b>"),
    w_err_title, w_err_txt, w_err_col,
    widgets.HTML("<hr><b>✅ Solution Section</b>"),
    w_opt_title, w_opt_txt, w_opt_col, w_body_font,
    widgets.HTML("<hr><b>📊 Stats / Number Block</b>"),
    w_show_stats, w_stats_y, w_stats_col,
    widgets.HBox([w_stats_num1, w_stats_lbl1]), widgets.HBox([w_stats_num2, w_stats_lbl2]), widgets.HBox([w_stats_num3, w_stats_lbl3]),
])

# Tab 6 – Layout, Divider & QR
tab6 = widgets.VBox([
    widgets.HTML("<b>📏 Spacing & Margin</b>"),
    w_margin_x, w_spacing,
    widgets.HTML("<hr><b>➖ Divider Line</b>"),
    w_show_divider, widgets.HBox([w_divider_col, w_divider_style]),
    widgets.HTML("<hr><b>📲 QR Code (V4 Feature)</b>"),
    w_show_qr, w_qr_url, widgets.HBox([w_qr_size, w_qr_col]),
])

# Tab 7 – Footer & Logo
tab7 = widgets.VBox([
    widgets.HTML("<b>🔻 Footer</b>"),
    w_show_footer, w_foot_txt, widgets.HBox([w_foot_sz, w_foot_col]),
    widgets.HTML("<hr><b>🖼️ Logo / Watermark</b>"),
    w_show_logo, w_logo_upload, widgets.HBox([w_logo_size, w_logo_opacity]), w_logo_pos,
])

# Tab 8 – Export
tab8 = widgets.VBox([
    widgets.HTML("<b>📤 Export Options</b>"),
    w_export_fmt, w_export_quality, w_filename,
])

ui_tabs = widgets.Tab(children=[tab1,tab2,tab3,tab4,tab5,tab6,tab7,tab8])
for i,n in enumerate(["📐 Canvas", "🎨 Background", "✍️ Header/Hero", "📦 Module", "📝 Content/Stats", "📏 Layout/QR", "🔻 Footer/Logo", "📤 Export"]):
    ui_tabs.set_title(i, n)

print("✅ Step 7: UI Tabs Organization Ready!")

✅ Step 7: UI Tabs Organization Ready!


In [10]:
# @title 🔄 Step 8: Live Engine + Undo + Save/Load

preview_output  = widgets.Output()
current_image   = None
_is_rendering   = False
_undo_stack     = deque(maxlen=5)

w_preview_size  = widgets.IntSlider(
    value=420, min=150, max=900, step=25,
    description="🔍 Preview সাইজ:", style={"description_width":"130px"}, layout=widgets.Layout(width="500px"))

widget_mapping = {
    # Canvas & BG
    "cv_w":w_cv_w, "cv_h":w_cv_h, "bg_start":w_bg_start, "bg_end":w_bg_end, "grad_dir":w_grad_dir,
    "show_accent_bar":w_show_accent_bar, "accent_bar_h":w_accent_bar_h, "accent_bar_col":w_accent_bar_col,
    "bg_img_opacity":w_bg_img_opacity,
    "show_pattern":w_show_pattern, "pattern_type":w_pattern_type, "pattern_col":w_pattern_col,
    "pattern_opacity":w_pattern_opacity, "pattern_spacing":w_pattern_spacing,
    "use_noise":w_use_noise, "noise_opacity":w_noise_opacity,

    # Header & Hero
    "head_txt":w_head_txt, "head_sz":w_head_sz, "head_col":w_head_col, "head_y":w_head_y,
    "head_align":w_head_align, "head_font":w_head_font,
    "hero1_txt":w_hero1_txt, "hero1_sz":w_hero1_sz, "hero1_col":w_hero1_col, "hero1_y":w_hero1_y,
    "hero2_txt":w_hero2_txt, "hero2_sz":w_hero2_sz, "hero2_col":w_hero2_col, "hero2_y":w_hero2_y,
    "show_hero3":w_show_hero3, "hero3_txt":w_hero3_txt, "hero3_sz":w_hero3_sz, "hero3_col":w_hero3_col,
    "hero3_y":w_hero3_y, "hero_shadow":w_hero_shadow, "hero_align":w_hero_align, "hero_font":w_hero_font,

    # Module & Badge
    "show_mod":w_show_mod, "module_count":w_module_count, "use_glass":w_use_glass, "glass_opacity":w_glass_opacity,
    "mod_y":w_mod_y, "mod_h":w_mod_h, "mod_bg":w_mod_bg, "mod_rad":w_mod_rad,
    "show_mod_border":w_show_mod_border, "mod_border_col":w_mod_border_col, "mod_shadow":w_mod_shadow,
    "bdg_num":w_bdg_num, "bdg_title":w_bdg_title, "bdg_sz":w_bdg_sz, "bdg_col":w_bdg_col, "bdg_shape":w_bdg_shape,

    # Content & Stats
    "err_title":w_err_title, "err_txt":w_err_txt, "err_col":w_err_col,
    "opt_title":w_opt_title, "opt_txt":w_opt_txt, "opt_col":w_opt_col, "body_font":w_body_font,
    "show_stats":w_show_stats, "stats_y":w_stats_y, "stats_col":w_stats_col,
    "stats_num1":w_stats_num1, "stats_lbl1":w_stats_lbl1,
    "stats_num2":w_stats_num2, "stats_lbl2":w_stats_lbl2,
    "stats_num3":w_stats_num3, "stats_lbl3":w_stats_lbl3,

    # QR, Logo, Footer, Layout
    "show_qr":w_show_qr, "qr_url":w_qr_url, "qr_size":w_qr_size, "qr_col":w_qr_col,
    "show_logo":w_show_logo, "logo_size":w_logo_size, "logo_pos":w_logo_pos, "logo_opacity":w_logo_opacity,
    "show_footer":w_show_footer, "foot_txt":w_foot_txt, "foot_col":w_foot_col, "foot_sz":w_foot_sz,
    "margin_x":w_margin_x, "spacing":w_spacing,
    "show_divider":w_show_divider, "divider_col":w_divider_col, "divider_style":w_divider_style,

    # Export
    "export_fmt":w_export_fmt, "export_quality":w_export_quality, "filename":w_filename,
}

def _collect_values(): return {k: w.value for k, w in widget_mapping.items()}

def _do_render(vals=None):
    global current_image, _is_rendering
    if _is_rendering: return
    _is_rendering = True
    try:
        vals = vals or _collect_values()
        bg_pil   = get_uploaded_image(w_bg_img_upload)
        logo_pil = get_uploaded_image(w_logo_upload)

        with preview_output:
            clear_output(wait=True)
            img, ms = render_template(vals, bg_pil, logo_pil)
            psize  = w_preview_size.value
            aspect = img.height / img.width
            thumb  = img.resize((psize, int(psize*aspect)), Image.LANCZOS)
            display(thumb)
            display(HTML(f"<div style='color:#607D8B;font-size:12px;margin-top:4px;'>⏱️ {ms:.0f}ms &nbsp;|&nbsp; 📐 {img.width}×{img.height}px</div>"))
            current_image = img
    finally:
        _is_rendering = False

def _on_change(change): _do_render()

def _on_preview_size_change(change):
    global current_image
    if current_image is None: return
    with preview_output:
        clear_output(wait=True)
        psize  = w_preview_size.value
        aspect = current_image.height / current_image.width
        thumb  = current_image.resize((psize, int(psize*aspect)), Image.LANCZOS)
        display(thumb)
        display(HTML(f"<div style='color:#607D8B;font-size:12px;'>📐 {current_image.width}×{current_image.height}px &nbsp;|&nbsp; 🔍 {thumb.width}×{thumb.height}px</div>"))

for wgt in widget_mapping.values(): wgt.observe(_on_change, names="value")
w_bg_img_upload.observe(lambda c: _do_render() if c["name"]=="value" else None, names="value")
w_logo_upload.observe(lambda c: _do_render() if c["name"]=="value" else None, names="value")
w_preview_size.observe(_on_preview_size_change, names="value")

def _save_snapshot(): _undo_stack.append({k: w.value for k, w in widget_mapping.items()})
def _restore_snapshot():
    if not _undo_stack: return False
    state = _undo_stack.pop()
    for k, v in state.items():
        w = widget_mapping.get(k)
        if w is not None:
            try: w.value = v
            except Exception: pass
    return True

def _save_json():
    data = _collect_values()
    serializable = {}
    for k, v in data.items():
        try:
            _json.dumps(v)
            serializable[k] = v
        except Exception:
            serializable[k] = str(v)
    fname = "template_v4_settings.json"
    with open(fname, "w", encoding="utf-8") as f:
        _json.dump(serializable, f, ensure_ascii=False, indent=2)
    return fname

def _load_json_from_bytes(content_bytes):
    data = _json.loads(content_bytes.decode("utf-8"))
    for k, v in data.items():
        w = widget_mapping.get(k)
        if w is None: continue
        try:
            cur = w.value
            if isinstance(cur, bool):   w.value = bool(v)
            elif isinstance(cur, int):  w.value = int(v)
            elif isinstance(cur, float):w.value = float(v)
            else:                       w.value = str(v)
        except Exception: pass

print("✅ Step 8: Live Engine + Undo + Save/Load Ready!")

✅ Step 8: Live Engine + Undo + Save/Load Ready!


In [11]:
# @title 📥 Step 9: Download, Reset & Action Buttons

try:
    from google.colab import files
except ImportError:
    pass # Fallback for local Jupyter environments

DEFAULTS = {
    "cv_w":1080,"cv_h":1350,"bg_start":"#050814","bg_end":"#1A1030","grad_dir":"vertical",
    "show_accent_bar":True,"accent_bar_h":8,"accent_bar_col":"#00E5FF",
    "bg_img_opacity":0.4,
    "show_pattern":False,"pattern_type":"dots","pattern_col":"#FFFFFF","pattern_opacity":0.08,"pattern_spacing":40,
    "use_noise":False, "noise_opacity":0.04,
    "head_txt":"SYSTEM PROTOCOL // ANALYSIS","head_sz":20,"head_col":"#00E5FF","head_y":60,"head_align":"left","head_font":"Regular",
    "hero1_txt":"কেন ছাত্রছাত্রীরা ভুল","hero1_sz":90,"hero1_col":"#FFFFFF","hero1_y":150,
    "hero2_txt":"পদ্ধতিতে পড়াশোনা করে?","hero2_sz":90,"hero2_col":"#00E5FF","hero2_y":270,
    "show_hero3":False,"hero3_txt":"","hero3_sz":70,"hero3_col":"#B0BEC5","hero3_y":390,
    "hero_shadow":True,"hero_align":"left","hero_font":"Bold",
    "show_mod":True,"module_count":1,"use_glass":False,"glass_opacity":0.3,"mod_y":450,"mod_h":420,"mod_bg":"#121826","mod_rad":30,
    "show_mod_border":True,"mod_border_col":"#00E5FF","mod_shadow":True,
    "bdg_num":"01","bdg_title":"রিটেনশন ফ্যাক্টর","bdg_sz":24,"bdg_col":"#00E5FF","bdg_shape":"circle",
    "err_title":"DETECTED ERROR:","err_txt":"মনে মনে পড়া বা প্যাসিভ লার্নিং।","err_col":"#FF5722",
    "opt_title":"OPTIMAL PATH:","opt_txt":"অ্যাক্টিভ রিকল এবং স্পেসড রিপিটেশন।","opt_col":"#00C853",
    "body_font":"Medium",
    "show_stats":False,"stats_y":1050,"stats_col":"#00E5FF",
    "stats_num1":"৯৫%","stats_lbl1":"সাফল্যের হার","stats_num2":"৪৮H","stats_lbl2":"কোর্স সময়","stats_num3":"১০K+","stats_lbl3":"শিক্ষার্থী",
    "show_qr":False,"qr_url":"https://yourwebsite.com","qr_size":150,"qr_col":"#FFFFFF",
    "show_logo":False,"logo_size":100,"logo_pos":"top-right","logo_opacity":1.0,
    "show_footer":True,"foot_txt":"© Your Name | your.website.com","foot_col":"#607D8B","foot_sz":22,
    "margin_x":80,"spacing":45,
    "show_divider":True,"divider_col":"#00E5FF","divider_style":"solid",
    "export_fmt":"PNG","export_quality":92,"filename":"my_template_v4",
}

out_log = widgets.Output()

# Buttons
btn_download = widgets.Button(description="📥 Download Image",  button_style="success", layout=widgets.Layout(width="220px",height="48px"))
btn_reset    = widgets.Button(description="🔄 Reset Defaults",  button_style="warning", layout=widgets.Layout(width="180px",height="48px"))
btn_snapshot = widgets.Button(description="📷 Snapshot (Undo)", button_style="info",    layout=widgets.Layout(width="200px",height="48px"))
btn_undo     = widgets.Button(description="↩️ Undo",            button_style="danger",  layout=widgets.Layout(width="120px",height="48px"))
btn_save_json= widgets.Button(description="💾 Save Settings",   button_style="primary", layout=widgets.Layout(width="180px",height="48px"))
w_json_upload= widgets.FileUpload(accept=".json", multiple=False, description="📂 Load Settings:", layout=widgets.Layout(width="220px"))

def on_download(b):
    with out_log:
        clear_output()
        if current_image is None:
            print("❌ Preview নেই — কোনো slider পরিবর্তন করুন।"); return
        fmt  = w_export_fmt.value
        q    = w_export_quality.value
        name = (w_filename.value.strip() or "template_v4") + "." + fmt.lower().replace("jpeg","jpg")
        kw   = {"format": fmt}
        if fmt in ("JPEG","WEBP"): kw["quality"] = q
        try:
            current_image.save(name, **kw)
            try:
                files.download(name)
                print(f"✅ {name} ডাউনলোড হচ্ছে!")
            except:
                print(f"✅ {name} লোকাল ডিরেক্টরিতে সেভ হয়েছে!")
        except Exception as e:
            print(f"❌ {e}")

def on_reset(b):
    _save_snapshot()
    for k, v in DEFAULTS.items():
        w = widget_mapping.get(k)
        if w:
            try: w.value = v
            except Exception: pass
    with out_log:
        clear_output()
        print("✅ Defaults restored! (আগের state undo-তে আছে)")

def on_snapshot(b):
    _save_snapshot()
    with out_log:
        clear_output()
        print(f"📷 Snapshot saved! (stack: {len(_undo_stack)}/5)")

def on_undo(b):
    if _restore_snapshot():
        with out_log:
            clear_output()
            print(f"↩️ Undo done! (remaining: {len(_undo_stack)})")
    else:
        with out_log:
            clear_output()
            print("⚠️ Undo stack খালি।")

def on_save_json(b):
    try:
        fname = _save_json()
        try:
            files.download(fname)
            with out_log:
                clear_output()
                print(f"✅ {fname} saved & downloading!")
        except:
            with out_log:
                clear_output()
                print(f"✅ {fname} saved to local directory!")
    except Exception as e:
        with out_log:
            clear_output()
            print(f"❌ {e}")

def on_json_upload(change):
    if not w_json_upload.value: return
    try:
        val = w_json_upload.value
        content = list(val.values())[0]["content"] if isinstance(val,dict) else val[0]["content"]
        _save_snapshot()
        _load_json_from_bytes(content)
        with out_log:
            clear_output()
            print("✅ Settings loaded! (আগের state undo-তে আছে)")
    except Exception as e:
        with out_log:
            clear_output()
            print(f"❌ Load error: {e}")

btn_download.on_click(on_download)
btn_reset.on_click(on_reset)
btn_snapshot.on_click(on_snapshot)
btn_undo.on_click(on_undo)
btn_save_json.on_click(on_save_json)
w_json_upload.observe(on_json_upload, names="value")

print("✅ Step 9: Download + Action Buttons Ready!")

✅ Step 9: Download + Action Buttons Ready!


In [14]:
# @title 🚀 Step 10 (Professional UI Redesign): Advanced Studio Interface

from ipywidgets import Layout, VBox, HBox, Label, HTML, Checkbox, Button, Output, Accordion, Tab, GridspecLayout
import ipywidgets as widgets

# ==================== CUSTOM STYLES ====================
display(HTML("""
<style>
:root {
    --bg-deep: #0b0f1a;
    --bg-card: #141b2b;
    --bg-sidebar: #0e1422;
    --accent-primary: #00e5ff;
    --accent-secondary: #ffd700;
    --text-light: #ffffff;
    --text-muted: #8f9bb3;
    --border-color: #2d3748;
    --success: #4caf50;
    --warning: #ff9800;
    --danger: #f44336;
}
body {
    background-color: var(--bg-deep);
    color: var(--text-light);
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
}
.widget-label, .widget-label-text {
    color: var(--text-muted) !important;
    font-size: 12px !important;
    font-weight: 500 !important;
    letter-spacing: 0.3px;
}
.widget-readout {
    color: var(--accent-primary) !important;
    font-weight: 600;
}
.bq-widget, .widget-slider, .widget-colorpicker, .widget-dropdown,
.widget-checkbox, .widget-text, .widget-textarea, .widget-intslider,
.widget-floatslider, .widget-toggleslider {
    background: var(--bg-card) !important;
    border: 1px solid var(--border-color) !important;
    border-radius: 10px !important;
    padding: 6px 12px !important;
    margin: 4px 0 !important;
    color: var(--text-light) !important;
    transition: all 0.2s ease;
}
.widget-button {
    background: linear-gradient(135deg, var(--bg-card) 0%, #1e2a3a 100%) !important;
    border: 1px solid var(--accent-primary) !important;
    color: var(--text-light) !important;
    border-radius: 24px !important;
    font-weight: 600 !important;
    font-size: 13px !important;
    padding: 8px 20px !important;
    box-shadow: 0 4px 6px rgba(0,0,0,0.3);
    transition: all 0.2s ease;
}
.widget-button:hover {
    background: var(--accent-primary) !important;
    color: var(--bg-deep) !important;
    border-color: var(--accent-primary) !important;
    transform: translateY(-2px);
    box-shadow: 0 8px 12px rgba(0,229,255,0.2);
}
.widget-button:active {
    transform: translateY(0);
}
.accordion .p-accordion-header {
    background: var(--bg-sidebar) !important;
    border: 1px solid var(--border-color) !important;
    color: var(--accent-primary) !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
    padding: 14px !important;
    margin: 4px 0 !important;
    font-size: 14px;
}
.accordion .p-accordion-header:hover {
    background: #1f2a3f !important;
    border-color: var(--accent-primary) !important;
}
.accordion .p-accordion-content {
    background: transparent !important;
    border: none !important;
    padding: 8px 16px 16px 16px !important;
}
.tab .p-TabPanel-tab {
    background: var(--bg-card) !important;
    color: var(--text-muted) !important;
    border: 1px solid var(--border-color) !important;
}
.tab .p-TabPanel-tab.p-mod-current {
    background: var(--accent-primary) !important;
    color: var(--bg-deep) !important;
}
hr {
    border: none;
    height: 1px;
    background: linear-gradient(90deg, transparent, var(--accent-primary), transparent);
    margin: 20px 0;
}
.card {
    background: var(--bg-card);
    border-radius: 16px;
    padding: 20px;
    border: 1px solid var(--border-color);
    box-shadow: 0 10px 30px rgba(0,0,0,0.5);
}
.preview-container {
    background: var(--bg-deep);
    border-radius: 16px;
    padding: 15px;
    border: 2px solid var(--accent-primary);
    box-shadow: 0 0 30px rgba(0,229,255,0.2);
}
</style>
"""))

# ==================== HEADER ====================
header = HTML("""
<div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 24px;">
    <div>
        <h1 style="color: #00e5ff; font-size: 28px; margin: 0; font-weight: 700; letter-spacing: -0.5px;">
            🎨 Template Studio <span style="color: #4caf50; font-size: 16px; background: #1a2a3a; padding: 4px 12px; border-radius: 30px; margin-left: 12px;">v4 Professional</span>
        </h1>
        <p style="color: #8f9bb3; margin: 8px 0 0; font-size: 15px;">
            Glassmorphism · Multi-Module · QR · 32 Themes · 10 Full Templates · Undo/Redo
        </p>
    </div>
    <div style="background: #141b2b; border-radius: 50px; padding: 8px 20px; border: 1px solid #2d3748;">
        <span style="color: #00e5ff;">⚡ Live Preview</span>
    </div>
</div>
""")

# ==================== LEFT SIDEBAR ====================
# Preset sections
preset_section = VBox([
    HTML("<h3 style='color: #00e5ff; margin: 0 0 8px 0;'>🎨 Quick Color Themes</h3>"),
    preset_box,  # from Step 6 (32 themes)
    HTML("<h3 style='color: #00e5ff; margin: 20px 0 8px 0;'>📋 Full Template Styles</h3>"),
    style_preset_box,  # from Step 6c (10 styles)
], layout=Layout(margin="0 0 20px 0"))

# Main controls as accordion (categories)
accordion_categories = Accordion([
    canvas_box, bg_box, header_box, module_box, content_box, extra_box  # defined in previous step
], selected_index=None)
for i, title in enumerate(["📐 Canvas", "🎨 Background", "✍️ Header/Hero", "📦 Module", "📝 Content/Stats", "📏 Extras"]):
    accordion_categories.set_title(i, title)

left_sidebar = VBox([
    preset_section,
    HTML("<hr>"),
    HTML("<h3 style='color: #00e5ff; margin: 0 0 8px 0;'>⚙️ Advanced Controls</h3>"),
    accordion_categories
], layout=Layout(width="100%", padding="0 10px 0 0"))

# ==================== RIGHT PANEL (PREVIEW + ACTIONS) ====================
# Auto-render toggle and manual render button
auto_render_cb = Checkbox(value=True, description="🔄 Auto-render", style={'description_width': 'initial'}, layout=Layout(width="auto"))
render_btn = Button(description="🎯 Render Now", button_style='primary', layout=Layout(width="140px"))
render_output = Output()

# Live preview area
preview_header = HBox([
    HTML("<b style='color:#00e5ff;'>🖼️ Live Preview</b>"),
    HBox([auto_render_cb, render_btn], layout=Layout(justify_content="flex-end", flex="1"))
], layout=Layout(justify_content="space-between", align_items="center", margin="0 0 10px 0"))

preview_box = VBox([
    preview_header,
    HBox([w_preview_size,
          HTML("<span style='color:#8f9bb3; font-size:12px;'> (output resolution unchanged)</span>")],
         layout=Layout(align_items="center", margin="0 0 10px 0")),
    preview_output
], layout=Layout(padding="0"))

# Action buttons + export
action_buttons = HBox([
    btn_download, btn_snapshot, btn_undo, btn_reset, btn_save_json
], layout=Layout(justify_content="center", gap="8px", flex_wrap="wrap"))

upload_row = HBox([w_json_upload], layout=Layout(justify_content="center", margin="12px 0"))

export_options = HBox([
    w_export_fmt, w_export_quality, w_filename
], layout=Layout(justify_content="center", gap="15px", flex_wrap="wrap", margin="20px 0 10px 0"))

right_panel = VBox([
    preview_box,
    HTML("<hr>"),
    action_buttons,
    upload_row,
    HTML("<hr>"),
    HTML("<b style='color:#00e5ff;'>📤 Export Settings</b>"),
    export_options,
    out_log  # from Step 9
], layout=Layout(width="100%", padding="0 0 0 20px"))

# ==================== MAIN LAYOUT ====================
main_layout = HBox([
    left_sidebar,
    right_panel
], layout=Layout(justify_content="space-between", gap="20px"))

# ==================== OVERRIDE RENDER LOGIC FOR AUTO-RENDER ====================
# Replace the observe handlers to respect auto_render_cb
def conditional_render(change=None):
    if auto_render_cb.value:
        _do_render()

# Re-attach observers with conditional
for wgt in widget_mapping.values():
    wgt.unobserve(_on_change, names="value")
    wgt.observe(conditional_render, names="value")

w_bg_img_upload.unobserve(None, names="value")
w_bg_img_upload.observe(lambda c: conditional_render() if c["name"]=="value" else None, names="value")
w_logo_upload.unobserve(None, names="value")
w_logo_upload.observe(lambda c: conditional_render() if c["name"]=="value" else None, names="value")

def manual_render(b):
    _do_render()

render_btn.on_click(manual_render)

# ==================== FINAL DISPLAY ====================
display(header)
display(main_layout)

# Initial render
_do_render()

HTML(value="\n<style>\n:root {\n    --bg-deep: #0b0f1a;\n    --bg-card: #141b2b;\n    --bg-sidebar: #0e1422;\n…

HTML(value='\n<div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 2…